# 17 · FlashAttention 进阶:causal mask、GQA 与 Flash-Decoding

> Learn Triton 系列 · 阶段 3(推理优化)第 3 篇
> 前置:第 14 篇(FlashAttention 前向,必须先完成)、第 15 篇(prefill/decode 两形态)
> 运行环境:Google Colab T4 GPU

第 14 篇的教学版有三个"离生产差一步"的缺口:① 没有 **causal mask**(自回归模型必须的下三角);② 没有 **GQA**(现代模型 KV 头远少于 Q 头);③ 对 **decode 形态**(1 个 query 对几万 KV)完全无能为力。本篇逐个补齐,最后实现 **Flash-Decoding(split-K)**——decode 长上下文 attention 的标准解法,vLLM/SGLang/TensorRT-LLM 全部内置。

## 环境准备

In [ ]:
import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "请在 Colab 选择 GPU 运行时"

import triton
import triton.language as tl
import triton.testing

print(f"PyTorch {torch.__version__} | Triton {triton.__version__} | {torch.cuda.get_device_name(0)}")

---

## §1 是什么 & 能力边界

### 缺口一:causal mask 不只是 mask,更是"跳过一半计算"

自回归要求 token $i$ 只看 $j \le i$。在分块世界里,Q 块 $i$ 与 KV 块 $j$ 的关系分三类:

```text
j 整块 < i 块起点   -> 全可见:正常算,无需 mask        (约一半的块)
j 整块 > i 块终点   -> 全不可见:整块跳过,连 load 都不要!(约一半的块)
对角线附近          -> 部分可见:逐元素 mask             (只有 O(N) 个块)
```

正确实现的 causal attention 不是"算完再扔",而是 **KV 循环上界 = Q 块终点**:计算量直接减半。FlashAttention-2 的一个重要工程改进就是把对角块与全可见块分开处理,避免在多数块上做无谓的 `tl.where`。

### 缺口二:GQA(Grouped-Query Attention)

$H_q$ 个 Q 头分组共享 $H_{kv}$ 个 KV 头(Llama-3-8B:32/8;Qwen2.5-0.5B:14/2)。动机纯粹是**推理侧**的:KV cache 大小 ∝ $H_{kv}$(第 15 篇公式),压 4~7 倍。kernel 视角只改一行:**用 `q_head // group_size` 找到对应的 KV 头**——但绝不能在显存里物化 `repeat_interleave`(那等于放弃全部节省)。

### 缺口三:decode 没有 M 维并行 —— Flash-Decoding(split-K)

decode 时 Q 只有 1 行。第 14 篇的并行划分 `grid=(Q块数, BH)` 退化成 `grid=(1, BH)`:**batch×heads 个 program 而已**——BH=8 时,T4 的 40 个 SM 大半在围观。KV 越长,这条"独木桥"越慢。

解法用的是第 09 篇练习 2 埋的伏笔:**online softmax 统计量满足结合律,可以分段算再合并**。

```text
把 KV 切成 S 段,grid = (BH, S):每段独立算出
    (m_s, l_s, acc_s)        ← 各段以自己的 m_s 为基准,互不通信
第二步(很小的 merge):
    m* = max_s m_s
    l* = Σ l_s · e^{m_s - m*}
    O  = Σ acc_s · e^{m_s - m*} / l*
```

并行度从 BH 涨到 BH×S,SM 重新喂饱。这就是 Flash-Decoding(Dao 等,2023),也叫 split-K attention。

### 能做什么 / 不能做什么

能做:causal 减半计算;GQA 零开销支持;Flash-Decoding 把长上下文 decode 的 attention 延迟拉低数倍;三者可叠加。

不能做:

- Flash-Decoding 有**额外的 merge 开销**与 partial 缓冲写读:KV 短或 BH 大(本来就喂得饱)时,split 反而变慢——split 数要按"KV 长度 × BH vs SM 数"动态选(vLLM 内部就是启发式决定);
- decode 的 GEMV 形态用不上 Tensor Core(M=1 < 16,`tl.dot` 不可用),本篇用乘加+归约实现——没关系,decode attention 是 memory-bound,带宽才是天花板;
- 本篇仍是教学版:不含 paged KV(下一篇加上)、不含 FA2 的对角块特化、不含变长 batch。

---

## §2 递进式例子

### 例 1:causal FlashAttention —— 循环上界 + 对角块 mask

在第 14 篇 kernel 上改两处:KV 循环只走到本 Q 块终点;块内对 `m >= n` 做逐元素 mask。

In [ ]:
@triton.jit
def flash_attn_causal_kernel(
    q_ptr, k_ptr, v_ptr, o_ptr,
    stride_bh, stride_n, seqlen, sm_scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_bh = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, HEAD_DIM)
    base = pid_bh * stride_bh

    q = tl.load(q_ptr + base + offs_m[:, None] * stride_n + offs_d[None, :],
                mask=offs_m[:, None] < seqlen, other=0.0)
    m_i = tl.full((BLOCK_M,), float("-inf"), dtype=tl.float32)
    l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_M, HEAD_DIM), dtype=tl.float32)

    hi = (pid_m + 1) * BLOCK_M                  # ★ 改动 1:KV 只扫到 Q 块终点,后面整块跳过
    for start_n in range(0, hi, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)
        k_blk = tl.load(k_ptr + base + offs_n[:, None] * stride_n + offs_d[None, :],
                        mask=(offs_n[:, None] < seqlen), other=0.0)
        s = tl.dot(q, tl.trans(k_blk)) * sm_scale
        s = tl.where((offs_m[:, None] >= offs_n[None, :]) & (offs_n[None, :] < seqlen),
                     s, float("-inf"))           # ★ 改动 2:对角块的下三角 mask
        m_new = tl.maximum(m_i, tl.max(s, axis=1))
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(s - m_new[:, None])
        l_i = alpha * l_i + tl.sum(p, axis=1)
        v_blk = tl.load(v_ptr + base + offs_n[:, None] * stride_n + offs_d[None, :],
                        mask=(offs_n[:, None] < seqlen), other=0.0)
        acc = acc * alpha[:, None] + tl.dot(p.to(tl.float16), v_blk)
        m_i = m_new

    acc = acc / l_i[:, None]
    tl.store(o_ptr + base + offs_m[:, None] * stride_n + offs_d[None, :],
             acc.to(tl.float16), mask=offs_m[:, None] < seqlen)


def flash_causal(q, k, v):
    BH, N, D = q.shape
    o = torch.empty_like(q)
    grid = (triton.cdiv(N, 64), BH)
    flash_attn_causal_kernel[grid](q, k, v, o, q.stride(0), q.stride(1), N, D ** -0.5,
                                   BLOCK_M=64, BLOCK_N=64, HEAD_DIM=D,
                                   num_warps=4, num_stages=2)
    return o


BH, N, D = 8, 4096, 64
q = torch.randn(BH, N, D, device="cuda", dtype=torch.float16)
k, v = torch.randn_like(q), torch.randn_like(q)
torch.testing.assert_close(flash_causal(q, k, v),
                           F.scaled_dot_product_attention(q, k, v, is_causal=True),
                           rtol=1e-2, atol=1e-2)

ms_causal = triton.testing.do_bench(lambda: flash_causal(q, k, v), return_mode="median")
print(f"causal 正确 ✓;耗时 {ms_causal:.3f} ms")
print("跳过了约一半 KV 块 -> 相比全量版约 ~2x 省时(下三角只有一半面积)")

### 例 2:GQA —— 一行除法,不物化 repeat

In [ ]:
@triton.jit
def flash_gqa_kernel(
    q_ptr, k_ptr, v_ptr, o_ptr,
    q_stride_h, kv_stride_h, stride_n, seqlen, sm_scale,
    GROUP: tl.constexpr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_qh = tl.program_id(1)                  # q 头编号
    kv_h = pid_qh // GROUP                     # ★ GQA 的全部秘密:映射到共享的 KV 头
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, HEAD_DIM)

    q = tl.load(q_ptr + pid_qh * q_stride_h + offs_m[:, None] * stride_n + offs_d[None, :],
                mask=offs_m[:, None] < seqlen, other=0.0)
    m_i = tl.full((BLOCK_M,), float("-inf"), dtype=tl.float32)
    l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_M, HEAD_DIM), dtype=tl.float32)
    kb = k_ptr + kv_h * kv_stride_h
    vb = v_ptr + kv_h * kv_stride_h
    for start_n in range(0, seqlen, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)
        kv_mask = offs_n < seqlen
        k_blk = tl.load(kb + offs_n[:, None] * stride_n + offs_d[None, :],
                        mask=kv_mask[:, None], other=0.0)
        s = tl.dot(q, tl.trans(k_blk)) * sm_scale
        s = tl.where(kv_mask[None, :], s, float("-inf"))
        m_new = tl.maximum(m_i, tl.max(s, axis=1))
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(s - m_new[:, None])
        l_i = alpha * l_i + tl.sum(p, axis=1)
        v_blk = tl.load(vb + offs_n[:, None] * stride_n + offs_d[None, :],
                        mask=kv_mask[:, None], other=0.0)
        acc = acc * alpha[:, None] + tl.dot(p.to(tl.float16), v_blk)
        m_i = m_new
    tl.store(o_ptr + pid_qh * q_stride_h + offs_m[:, None] * stride_n + offs_d[None, :],
             (acc / l_i[:, None]).to(tl.float16), mask=offs_m[:, None] < seqlen)


Hq, Hkv, N, D = 16, 4, 2048, 64
q = torch.randn(Hq, N, D, device="cuda", dtype=torch.float16)
k = torch.randn(Hkv, N, D, device="cuda", dtype=torch.float16)
v = torch.randn_like(k)
o = torch.empty_like(q)
flash_gqa_kernel[(triton.cdiv(N, 64), Hq)](q, k, v, o, q.stride(0), k.stride(0), q.stride(1),
                                           N, D ** -0.5, GROUP=Hq // Hkv,
                                           BLOCK_M=64, BLOCK_N=64, HEAD_DIM=D, num_warps=4)
# 参考:显存里物化 repeat(我们 kernel 里避免的事)
ref = F.scaled_dot_product_attention(q, k.repeat_interleave(4, 0), v.repeat_interleave(4, 0))
torch.testing.assert_close(o, ref, rtol=1e-2, atol=1e-2)
print("GQA 正确 ✓ —— KV 在显存中只有 1/4 份,kernel 内用头映射共享,无任何物化")

### 例 3:Flash-Decoding —— split-K 两步走

decode 形态:每个 (batch, head) 只有 1 个 query。第一步 kernel 按 KV 段并行产出 partial;第二步在 torch 里 merge(段数很小,merge 開銷可忽略)。

In [ ]:
@triton.jit
def flash_decode_split_kernel(
    q_ptr, k_ptr, v_ptr,                       # q: [BH, D], k/v: [BH, S, D]
    m_ptr, l_ptr, acc_ptr,                     # partial: [BH, SPLITS], [BH, SPLITS], [BH, SPLITS, D]
    stride_bh, stride_n, seqlen, sm_scale,
    SPLITS: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    pid_bh = tl.program_id(0)
    pid_s = tl.program_id(1)                   # 第几段 KV
    offs_d = tl.arange(0, HEAD_DIM)

    q = tl.load(q_ptr + pid_bh * HEAD_DIM + offs_d).to(tl.float32)   # 单行 query

    chunk = tl.cdiv(seqlen, SPLITS)
    lo = pid_s * chunk
    hi = tl.minimum(lo + chunk, seqlen)

    m = float("-inf")
    l = 0.0
    acc = tl.zeros((HEAD_DIM,), dtype=tl.float32)
    for start in range(lo, hi, BLOCK_N):
        offs_n = start + tl.arange(0, BLOCK_N)
        mask_n = offs_n < hi
        k_blk = tl.load(k_ptr + pid_bh * stride_bh + offs_n[:, None] * stride_n + offs_d[None, :],
                        mask=mask_n[:, None], other=0.0).to(tl.float32)
        s = tl.sum(q[None, :] * k_blk, axis=1) * sm_scale        # GEMV:M=1 用乘加归约
        s = tl.where(mask_n, s, float("-inf"))
        m_new = tl.maximum(m, tl.max(s, axis=0))
        alpha = tl.exp(m - m_new)
        p = tl.exp(s - m_new)
        l = alpha * l + tl.sum(p, axis=0)
        v_blk = tl.load(v_ptr + pid_bh * stride_bh + offs_n[:, None] * stride_n + offs_d[None, :],
                        mask=mask_n[:, None], other=0.0).to(tl.float32)
        acc = acc * alpha + tl.sum(p[:, None] * v_blk, axis=0)
        m = m_new

    tl.store(m_ptr + pid_bh * SPLITS + pid_s, m)
    tl.store(l_ptr + pid_bh * SPLITS + pid_s, l)
    tl.store(acc_ptr + (pid_bh * SPLITS + pid_s) * HEAD_DIM + offs_d, acc)


def flash_decoding(q, k, v, splits=8, BLOCK_N=128):
    BH, S, D = k.shape
    m = torch.empty(BH, splits, device="cuda", dtype=torch.float32)
    l = torch.empty(BH, splits, device="cuda", dtype=torch.float32)
    acc = torch.empty(BH, splits, D, device="cuda", dtype=torch.float32)
    flash_decode_split_kernel[(BH, splits)](q, k, v, m, l, acc,
                                            k.stride(0), k.stride(1), S, D ** -0.5,
                                            SPLITS=splits, BLOCK_N=BLOCK_N, HEAD_DIM=D,
                                            num_warps=4)
    # ---- merge(第 09 篇练习 2 的合并公式)----
    m_star = m.max(dim=1, keepdim=True).values                  # [BH, 1]
    scale = torch.exp(m - m_star)                               # [BH, SPLITS]
    l_star = (l * scale).sum(dim=1, keepdim=True)               # [BH, 1]
    out = (acc * scale[..., None]).sum(dim=1) / l_star          # [BH, D]
    return out.half()


BH, S, D = 8, 32768, 64
qd = torch.randn(BH, D, device="cuda", dtype=torch.float16)
kd = torch.randn(BH, S, D, device="cuda", dtype=torch.float16)
vd = torch.randn_like(kd)
ref = F.scaled_dot_product_attention(qd[:, None, None, :], kd[:, None, :, :], vd[:, None, :, :])[:, 0, 0, :]
torch.testing.assert_close(flash_decoding(qd, kd, vd), ref, rtol=2e-2, atol=2e-2)
print(f"Flash-Decoding 正确 ✓ (BH={BH}, KV 长度 {S}, 8 段并行 + merge)")

---

## §3 知识连接

**与前面篇章:**

- 例 3 的 merge 公式 = 第 09 篇"统计量结合律"练习的工业落地;`s = tl.sum(q*k, axis=1)` 的 GEMV 写法呼应第 10 篇"tl.dot 维度 ≥16"的边界;
- causal 的"整块跳过"是第 04 篇 mask 哲学的升级:**最好的 mask 是连算都不算**;
- GQA 的头映射就是第 05 篇 gather 思想用在"头"这个维度上。

**与真实框架:**

- vLLM:v1 引擎的 decode attention 内置 split-KV 逻辑(FlashAttention 后端的 `num_splits` 参数);Triton 后端 `vllm/attention/ops/` 下可找到与例 3 同构的 split + merge 两段式;
- SGLang 的 decode attention kernel(`sgl-kernel`)同样是 split-K 结构;TensorRT-LLM 的 XQA kernel 是其闭源高度优化版;
- FA 演进脉络(面试常考):**FA1**(2022)tiling+online softmax 奠基;**FA2**(2023)Q 块外层并行、减少非 matmul 操作、更好的 warp 划分,2~3x 于 FA1;**Flash-Decoding**(2023)KV 维 split 解决 decode 并行度;**FA3**(2024)Hopper 特化:TMA 异步搬运、warp specialization、FP8。算法核心三代未变,变的全是"怎么喂饱新硬件"。

---

## §4 闭环对比实验:decode attention 的并行度危机与解药

固定 BH=8(小 batch 长上下文的典型困境),KV 长度 2K→64K,对比:① 不切分(splits=1,即"独木桥"版);② Flash-Decoding(splits=8);③ SDPA 参考。

In [ ]:
import matplotlib.pyplot as plt

kv_lens = [2048, 8192, 16384, 32768, 65536]
res = {"no-split (splits=1)": [], "Flash-Decoding (splits=8)": [], "SDPA": []}

for S in kv_lens:
    kk = torch.randn(BH, S, D, device="cuda", dtype=torch.float16)
    vv = torch.randn_like(kk)
    ref = F.scaled_dot_product_attention(qd[:, None, None, :], kk[:, None, :, :], vv[:, None, :, :])[:, 0, 0, :]
    torch.testing.assert_close(flash_decoding(qd, kk, vv, splits=8), ref, rtol=2e-2, atol=2e-2)

    for name, fn in [
        ("no-split (splits=1)", lambda: flash_decoding(qd, kk, vv, splits=1)),
        ("Flash-Decoding (splits=8)", lambda: flash_decoding(qd, kk, vv, splits=8)),
        ("SDPA", lambda: F.scaled_dot_product_attention(
            qd[:, None, None, :], kk[:, None, :, :], vv[:, None, :, :])),
    ]:
        ms = triton.testing.do_bench(fn, return_mode="median")
        res[name].append(ms)

print(f"{'KV 长度':>8} | " + " | ".join(f"{k:>26}" for k in res))
for i, S in enumerate(kv_lens):
    print(f"{S:>8} | " + " | ".join(f"{res[k][i]:>23.3f} ms" for k in res))

plt.figure(figsize=(9, 4.5))
for name, vals in res.items():
    plt.loglog(kv_lens, vals, "o-", label=name)
plt.xlabel("KV length"); plt.ylabel("decode attention time (ms)")
plt.title(f"Decode attention, batch x heads = {BH} (T4, 40 SMs)")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.show()

bw_check = (BH * kv_lens[-1] * D * 2 * 2) / (res["Flash-Decoding (splits=8)"][-1] / 1000) / 1e9
print(f"\nsanity check: 64K 时 Flash-Decoding 的 KV 读取带宽 ≈ {bw_check:.0f} GB/s(贴近 320 即达到物理上限)")

### 实验结果解读

- **no-split 版**随 KV 线性变慢且斜率最陡:8 个 program 串行啃长 KV,32 个 SM 闲置——并行度不足的代价一目了然;
- **Flash-Decoding** 把同样的活摊给 64 个 program,长 KV 下数倍领先,带宽逼近 320GB/s 物理上限——decode attention 是 memory-bound,**把带宽打满即是终点**;
- 短 KV 时两者差距缩小甚至反转(partial 缓冲与 merge 的固定开销占比上升)——印证 §1"split 数要动态选"的边界;
- 把这张图与第 14 篇的图放在一起,attention 的两个形态(prefill: M 维并行 / decode: KV 维并行)就都有了答案;下一篇把 KV 从"连续大张量"换成"分页",拼上推理 attention 的最后一块。

---

## §5 练习 + 面试考点

### 动手练习

1. 给例 3 加 **GQA 支持**(例 2 的头映射搬过来),并把 merge 也写成 Triton kernel,做到端到端无 torch 参与。
2. 扫 splits ∈ {1,2,4,8,16,32},在 KV=8K/64K 两档下找最优值,总结"最优 splits ≈ f(SM 数 / BH, KV 长度)"的经验关系——这就是框架里启发式的来历。

### 面试高频考点

- **Q:causal attention 在分块实现里怎么处理才高效?**
  A:三类块分治:全可见块正常算、全遮蔽块直接跳过(循环上界控制,连 load 都省)、仅对角块做逐元素 mask。计算量减半,而不是"算完再丢"。
- **Q:GQA 为什么能省 KV cache?kernel 里怎么支持?**
  A:多个 Q 头共享一个 KV 头,KV cache ∝ KV 头数,直接除以 group 数。kernel 里只是 `kv_head = q_head // group`,严禁物化 repeat_interleave(否则显存节省归零)。MQA 是 group=H 的极端;MLA(DeepSeek)走低秩压缩路线,进一步小但 kernel 更复杂。
- **Q:decode 阶段 FlashAttention 为什么失效?Flash-Decoding 怎么解决?**
  A:FA 的并行度来自 Q 块维度,decode 的 Q 只有 1 行,并行度退化为 batch×heads,SM 喂不饱。Flash-Decoding 沿 KV 维切成 S 段并行算 partial(m, l, acc),再用 online softmax 的结合律 merge——并行度 ×S,长 KV 下数倍提速;代价是 partial 读写与 merge,短 KV 不划算。
- **Q:FlashAttention 1/2/3 与 Flash-Decoding 分别解决什么?**
  A:FA1:IO 问题(tiling+online softmax);FA2:GPU 利用率(并行维度重排、减少非 matmul 开销);Flash-Decoding:decode 并行度(split-K);FA3:Hopper 硬件特性(TMA、warp specialization、FP8)。一句话:算法一次定型,后续全是面向硬件与形态的工程演化。